# Differential Evolution and Memetic Algorithms for Feature Selection

**Module 06 — Evolutionary & Swarm Methods**

Estimated time: 15 minutes

## What you will build

1. DE/rand/1 for feature selection with continuous relaxation
2. Memetic algorithm: DE + hill-climbing local search
3. Head-to-head benchmark: pure DE vs memetic DE vs GA vs PSO
4. Wall-clock time comparison

**References**:
- Storn & Price (1997). Differential Evolution — A Simple and Efficient Heuristic for Global Optimization over Continuous Spaces. *Journal of Global Optimization*, 11(4), 341–359.
- Moscato (1989). On Evolution, Search, Optimization, Genetic Algorithms and Martial Arts: Towards Memetic Algorithms. Caltech Technical Report.

In [ ]:
import time
import random as py_random
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ── Shared dataset ─────────────────────────────────────────────────────────────
data = load_breast_cancer()
X_raw, y = data.data, data.target
X = StandardScaler().fit_transform(X_raw)
N_FEATURES = X.shape[1]
FEATURE_NAMES = data.feature_names

CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Dataset: {X.shape[0]} samples, {N_FEATURES} features")

# Baseline
clf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
baseline_scores = cross_val_score(clf_base, X, y, cv=CV5, scoring='accuracy')
BASELINE_ACC = baseline_scores.mean()
BASELINE_ERR = 1.0 - BASELINE_ACC
print(f"Baseline (all {N_FEATURES} features): {BASELINE_ACC:.4f} accuracy")

# ── Shared evaluation function ────────────────────────────────────────────────
_eval_cache = {}

def evaluate_mask(mask: np.ndarray, n_trees: int = 50, n_splits: int = 3) -> float:
    """Cached cross-validation error for binary feature mask."""
    key = tuple(mask.astype(int))
    if key in _eval_cache:
        return _eval_cache[key]
    if not mask.any():
        _eval_cache[key] = 1.0
        return 1.0
    clf = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    err = 1.0 - cross_val_score(clf, X[:, mask], y, cv=cv, scoring='accuracy').mean()
    _eval_cache[key] = err
    return err


def continuous_to_binary(z: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Convert continuous DE vector to binary mask via sigmoid threshold."""
    prob = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
    mask = prob > threshold
    if not mask.any():
        mask[np.argmax(prob)] = True  # ensure at least one feature
    return mask


print("\nShared evaluation function ready (with caching).")

## 1. DE/rand/1 Implementation

Three key operations per individual per generation:
1. **Mutation**: `v = x_r1 + F * (x_r2 - x_r3)`
2. **Crossover**: binomial crossover with rate CR
3. **Selection**: greedy — trial replaces target only if better

In [ ]:
def de_rand_1(
    pop_size: int = 40,
    n_generations: int = 50,
    F: float = 0.5,
    CR: float = 0.7,
    init_range: float = 3.0,
    strategy: str = 'rand/1',
    random_state: int = 42,
) -> dict:
    """
    Differential Evolution for feature selection.

    Parameters
    ----------
    F : float
        Mutation scale factor (Storn & Price 1997, typically 0.4-0.9).
    CR : float
        Crossover rate (typically 0.7-0.9).
    strategy : 'rand/1', 'best/1', or 'current-to-best/1'
    init_range : float
        Continuous population initialised in [-init_range, +init_range].
        Sigmoid maps this to [0.05, 0.95] probability range.

    Returns
    -------
    dict with keys: best_mask, best_fitness, fitness_history,
                    feature_count_history, population, pop_fitness
    """
    rng = np.random.default_rng(random_state)
    start_time = time.time()

    # Initialise continuous population
    population = rng.uniform(-init_range, init_range, size=(pop_size, N_FEATURES))

    # Convert to binary and evaluate
    pop_masks = np.array([continuous_to_binary(ind) for ind in population])
    pop_fitness = np.array([evaluate_mask(mask) for mask in pop_masks])

    best_idx = np.argmin(pop_fitness)
    best_fitness = pop_fitness[best_idx]
    best_mask = pop_masks[best_idx].copy()

    fitness_history = [best_fitness]
    feature_count_history = [int(best_mask.sum())]

    for gen in range(n_generations):
        for i in range(pop_size):
            # Select base vector and difference vectors
            candidates = [j for j in range(pop_size) if j != i]

            if strategy == 'rand/1':
                r1, r2, r3 = rng.choice(candidates, size=3, replace=False)
                mutant = population[r1] + F * (population[r2] - population[r3])

            elif strategy == 'best/1':
                r1, r2 = rng.choice(candidates, size=2, replace=False)
                best_cont_idx = np.argmin(pop_fitness)
                mutant = (population[best_cont_idx]
                          + F * (population[r1] - population[r2]))

            elif strategy == 'current-to-best/1':
                r1, r2 = rng.choice(candidates, size=2, replace=False)
                best_cont_idx = np.argmin(pop_fitness)
                mutant = (population[i]
                          + F * (population[best_cont_idx] - population[i])
                          + F * (population[r1] - population[r2]))

            # Binomial crossover
            j_rand = rng.integers(N_FEATURES)  # at least one gene from mutant
            cross_mask = (rng.random(N_FEATURES) <= CR)
            cross_mask[j_rand] = True
            trial = np.where(cross_mask, mutant, population[i])

            # Evaluate trial
            trial_mask = continuous_to_binary(trial)
            trial_fit = evaluate_mask(trial_mask)

            # Greedy selection
            if trial_fit <= pop_fitness[i]:
                population[i] = trial
                pop_masks[i] = trial_mask
                pop_fitness[i] = trial_fit

        # Update global best
        current_best_idx = np.argmin(pop_fitness)
        if pop_fitness[current_best_idx] < best_fitness:
            best_fitness = pop_fitness[current_best_idx]
            best_mask = pop_masks[current_best_idx].copy()

        fitness_history.append(best_fitness)
        feature_count_history.append(int(best_mask.sum()))

    wall_time = time.time() - start_time
    return {
        'best_mask': best_mask,
        'best_fitness': best_fitness,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
        'wall_time': wall_time,
        'population': population,
        'pop_fitness': pop_fitness,
    }


print("DE/rand/1 implementation ready.")
print("Strategy options: 'rand/1', 'best/1', 'current-to-best/1'")

In [ ]:
print("Running DE/rand/1 (40 population, 50 generations, F=0.5, CR=0.7)...")
de_result = de_rand_1(
    pop_size=40, n_generations=50, F=0.5, CR=0.7, strategy='rand/1', random_state=42
)

# Validate with 5-fold CV
clf_de = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
de_scores = cross_val_score(clf_de, X[:, de_result['best_mask']], y,
                             cv=CV5, scoring='accuracy')

print(f"\nDE/rand/1 result:")
print(f"  3-fold CV error (training): {de_result['best_fitness']:.4f}")
print(f"  5-fold CV accuracy (validate): {de_scores.mean():.4f} ± {de_scores.std():.4f}")
print(f"  Features: {de_result['best_mask'].sum()}/{N_FEATURES}")
print(f"  Wall time: {de_result['wall_time']:.1f}s")
print(f"  Cache size: {len(_eval_cache)} unique evaluations")

## 2. Memetic Algorithm: DE + Hill Climbing

After each DE generation, apply a greedy bit-flip hill climb to every individual.
We use **Lamarckian learning**: the improved chromosome replaces the original in the population.

In [ ]:
def hill_climb_binary(
    mask: np.ndarray,
    current_fitness: float,
    max_steps: int = 30,
) -> tuple:
    """
    First-improvement greedy bit-flip hill climbing.

    Returns
    -------
    (improved_mask, improved_fitness, n_flips)
    """
    current = mask.copy()
    current_fit = current_fitness
    indices = list(range(N_FEATURES))
    n_flips = 0

    for _ in range(max_steps):
        # Random order for unbiased search
        np.random.shuffle(indices)
        improved = False

        for j in indices:
            neighbour = current.copy()
            neighbour[j] = not neighbour[j]
            if not neighbour.any():
                continue
            neighbour_fit = evaluate_mask(neighbour)
            if neighbour_fit < current_fit:
                current = neighbour
                current_fit = neighbour_fit
                n_flips += 1
                improved = True
                break  # first improvement

        if not improved:
            break

    return current, current_fit, n_flips


def memetic_de(
    pop_size: int = 40,
    n_generations: int = 50,
    F: float = 0.5,
    CR: float = 0.7,
    local_search_freq: int = 1,    # apply local search every N generations
    local_search_fraction: float = 0.3,  # fraction of pop to apply LS to
    max_ls_steps: int = 20,
    random_state: int = 42,
) -> dict:
    """
    Memetic Differential Evolution: DE/rand/1 + first-improvement hill climbing.

    Uses Lamarckian learning: improved chromosome replaces original in population.

    Parameters
    ----------
    local_search_freq : apply LS every this many generations
    local_search_fraction : fraction of individuals to refine with LS
    """
    rng = np.random.default_rng(random_state)
    np.random.seed(random_state)
    start_time = time.time()

    # Initialise population
    population = rng.uniform(-3.0, 3.0, size=(pop_size, N_FEATURES))
    pop_masks = np.array([continuous_to_binary(ind) for ind in population])
    pop_fitness = np.array([evaluate_mask(mask) for mask in pop_masks])

    best_idx = np.argmin(pop_fitness)
    best_fitness = pop_fitness[best_idx]
    best_mask = pop_masks[best_idx].copy()

    fitness_history = [best_fitness]
    feature_count_history = [int(best_mask.sum())]
    ls_improvements = []

    for gen in range(n_generations):
        # ── DE/rand/1 step ─────────────────────────────────────────────────────
        for i in range(pop_size):
            candidates = [j for j in range(pop_size) if j != i]
            r1, r2, r3 = rng.choice(candidates, size=3, replace=False)
            mutant = population[r1] + F * (population[r2] - population[r3])

            j_rand = rng.integers(N_FEATURES)
            cross_mask = (rng.random(N_FEATURES) <= CR)
            cross_mask[j_rand] = True
            trial = np.where(cross_mask, mutant, population[i])

            trial_mask = continuous_to_binary(trial)
            trial_fit = evaluate_mask(trial_mask)

            if trial_fit <= pop_fitness[i]:
                population[i] = trial
                pop_masks[i] = trial_mask
                pop_fitness[i] = trial_fit

        # ── Local search step (Lamarckian) ─────────────────────────────────────
        total_improvements = 0
        if (gen + 1) % local_search_freq == 0:
            n_ls = max(1, int(local_search_fraction * pop_size))
            # Apply LS to the best n_ls individuals
            ls_indices = np.argsort(pop_fitness)[:n_ls]

            for idx in ls_indices:
                improved_mask, improved_fit, n_flips = hill_climb_binary(
                    pop_masks[idx], pop_fitness[idx], max_steps=max_ls_steps
                )
                total_improvements += n_flips
                if improved_fit < pop_fitness[idx]:
                    # Lamarckian: update chromosome to improved version
                    # Map improved binary mask back to continuous space
                    pop_masks[idx] = improved_mask
                    pop_fitness[idx] = improved_fit
                    # Update continuous representation (project binary to logit space)
                    logit_improved = np.where(improved_mask, 2.0, -2.0)
                    population[idx] = logit_improved

        ls_improvements.append(total_improvements)

        # Update global best
        current_best_idx = np.argmin(pop_fitness)
        if pop_fitness[current_best_idx] < best_fitness:
            best_fitness = pop_fitness[current_best_idx]
            best_mask = pop_masks[current_best_idx].copy()

        fitness_history.append(best_fitness)
        feature_count_history.append(int(best_mask.sum()))

    wall_time = time.time() - start_time
    return {
        'best_mask': best_mask,
        'best_fitness': best_fitness,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
        'wall_time': wall_time,
        'ls_improvements': ls_improvements,
    }


print("Memetic DE implementation ready.")
print("Local search: first-improvement hill climbing, Lamarckian learning")

In [ ]:
print("Running Memetic DE (40 pop, 50 gen, LS on top 30% every generation)...")
memetic_result = memetic_de(
    pop_size=40,
    n_generations=50,
    F=0.5,
    CR=0.7,
    local_search_freq=1,
    local_search_fraction=0.3,
    max_ls_steps=20,
    random_state=42,
)

clf_mem = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
mem_scores = cross_val_score(clf_mem, X[:, memetic_result['best_mask']], y,
                              cv=CV5, scoring='accuracy')

print(f"\nMemetic DE result:")
print(f"  5-fold CV accuracy: {mem_scores.mean():.4f} ± {mem_scores.std():.4f}")
print(f"  Features: {memetic_result['best_mask'].sum()}/{N_FEATURES}")
print(f"  Wall time: {memetic_result['wall_time']:.1f}s")
print(f"  Total LS bit-flips: {sum(memetic_result['ls_improvements'])}")
print(f"  Cache size: {len(_eval_cache)} unique evaluations")

## 3. Run GA and PSO for Full Comparison

In [ ]:
def run_ga(pop_size=40, n_generations=50, random_state=42) -> dict:
    """Simple elitist GA for benchmark."""
    rng = py_random.Random(random_state)
    np_rng = np.random.default_rng(random_state)
    start_time = time.time()

    population = [np_rng.integers(0, 2, N_FEATURES).tolist() for _ in range(pop_size)]
    fit = [evaluate_mask(np.array(ind, dtype=bool)) for ind in population]

    best_fit = min(fit)
    best_ind = population[np.argmin(fit)].copy()
    fitness_history = [best_fit]
    feature_count_history = [sum(best_ind)]

    for _ in range(n_generations):
        new_pop = []
        for __ in range(pop_size):
            p1 = min(rng.sample(range(pop_size), 3), key=lambda i: fit[i])
            p2 = min(rng.sample(range(pop_size), 3), key=lambda i: fit[i])
            cx = rng.randint(1, N_FEATURES - 1)
            child = population[p1][:cx] + population[p2][cx:]
            child = [1 - b if rng.random() < 1.0 / N_FEATURES else b for b in child]
            if sum(child) == 0:
                child[rng.randint(0, N_FEATURES - 1)] = 1
            new_pop.append(child)

        new_fit = [evaluate_mask(np.array(ind, dtype=bool)) for ind in new_pop]
        all_pop = population + new_pop
        all_fit = fit + new_fit
        top = np.argsort(all_fit)[:pop_size]
        population = [all_pop[i] for i in top]
        fit = [all_fit[i] for i in top]

        if fit[0] < best_fit:
            best_fit = fit[0]
            best_ind = population[0].copy()

        fitness_history.append(best_fit)
        feature_count_history.append(sum(best_ind))

    wall_time = time.time() - start_time
    return {
        'best_mask': np.array(best_ind, dtype=bool),
        'best_fitness': best_fit,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
        'wall_time': wall_time,
    }


def run_pso(n_particles=40, n_iterations=50, random_state=42) -> dict:
    """Binary PSO (sigmoid) for benchmark."""
    rng = np.random.default_rng(random_state)
    start_time = time.time()
    V_MAX, W_MAX, W_MIN, C1, C2 = 4.0, 0.9, 0.4, 2.0, 2.0

    positions = rng.integers(0, 2, (n_particles, N_FEATURES)).astype(float)
    velocities = rng.uniform(-V_MAX, V_MAX, (n_particles, N_FEATURES))
    fitness = np.array([evaluate_mask(pos.astype(bool)) for pos in positions])
    pbest_pos = positions.copy()
    pbest_fit = fitness.copy()
    gbest_pos = positions[np.argmin(fitness)].copy()
    gbest_fit = fitness.min()
    fitness_history = [gbest_fit]
    feature_count_history = [int(gbest_pos.sum())]

    for t in range(n_iterations):
        w = W_MAX - (W_MAX - W_MIN) * t / n_iterations
        r1, r2 = rng.random((n_particles, N_FEATURES)), rng.random((n_particles, N_FEATURES))
        velocities = w * velocities + C1 * r1 * (pbest_pos - positions) + C2 * r2 * (gbest_pos - positions)
        velocities = np.clip(velocities, -V_MAX, V_MAX)
        probs = 1.0 / (1.0 + np.exp(-velocities))
        positions = (rng.random((n_particles, N_FEATURES)) < probs).astype(float)
        fitness = np.array([evaluate_mask(pos.astype(bool)) for pos in positions])
        improved = fitness < pbest_fit
        pbest_pos[improved] = positions[improved].copy()
        pbest_fit[improved] = fitness[improved]
        g_idx = np.argmin(pbest_fit)
        if pbest_fit[g_idx] < gbest_fit:
            gbest_fit = pbest_fit[g_idx]
            gbest_pos = pbest_pos[g_idx].copy()
        fitness_history.append(gbest_fit)
        feature_count_history.append(int(gbest_pos.sum()))

    wall_time = time.time() - start_time
    return {
        'best_mask': gbest_pos.astype(bool),
        'best_fitness': gbest_fit,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
        'wall_time': wall_time,
    }


print("Running GA...")
ga_result = run_ga(pop_size=40, n_generations=50, random_state=42)
clf_ga = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
ga_scores = cross_val_score(clf_ga, X[:, ga_result['best_mask']], y, cv=CV5, scoring='accuracy')
print(f"  GA: {ga_scores.mean():.4f} acc, {ga_result['best_mask'].sum()} feats, {ga_result['wall_time']:.1f}s")

print("Running PSO...")
pso_result = run_pso(n_particles=40, n_iterations=50, random_state=42)
clf_pso = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
pso_scores = cross_val_score(clf_pso, X[:, pso_result['best_mask']], y, cv=CV5, scoring='accuracy')
print(f"  PSO: {pso_scores.mean():.4f} acc, {pso_result['best_mask'].sum()} feats, {pso_result['wall_time']:.1f}s")

## 4. Head-to-Head Benchmark

In [ ]:
# Collect all results
clf_de_val = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
de_scores = cross_val_score(clf_de_val, X[:, de_result['best_mask']], y, cv=CV5, scoring='accuracy')

clf_mem_val = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
mem_scores = cross_val_score(clf_mem_val, X[:, memetic_result['best_mask']], y, cv=CV5, scoring='accuracy')

results = {
    'Baseline': {
        'acc': BASELINE_ACC,
        'std': baseline_scores.std(),
        'n_features': N_FEATURES,
        'wall_time': None,
        'history': [BASELINE_ERR] * 51,
    },
    'GA': {
        'acc': ga_scores.mean(),
        'std': ga_scores.std(),
        'n_features': int(ga_result['best_mask'].sum()),
        'wall_time': ga_result['wall_time'],
        'history': [1 - h for h in ga_result['fitness_history']],
    },
    'PSO': {
        'acc': pso_scores.mean(),
        'std': pso_scores.std(),
        'n_features': int(pso_result['best_mask'].sum()),
        'wall_time': pso_result['wall_time'],
        'history': [1 - h for h in pso_result['fitness_history']],
    },
    'DE/rand/1': {
        'acc': de_scores.mean(),
        'std': de_scores.std(),
        'n_features': int(de_result['best_mask'].sum()),
        'wall_time': de_result['wall_time'],
        'history': [1 - h for h in de_result['fitness_history']],
    },
    'Memetic DE': {
        'acc': mem_scores.mean(),
        'std': mem_scores.std(),
        'n_features': int(memetic_result['best_mask'].sum()),
        'wall_time': memetic_result['wall_time'],
        'history': [1 - h for h in memetic_result['fitness_history']],
    },
}

print(f"{'Method':<14} {'5-fold Acc':>12} {'Std':>7} {'Features':>10} {'Time (s)':>10}")
print("-" * 58)
for name, r in results.items():
    t = f"{r['wall_time']:.1f}" if r['wall_time'] else 'N/A'
    print(f"{name:<14} {r['acc']:>12.4f} {r['std']:>7.4f} {r['n_features']:>10} {t:>10}")

## 5. Convergence and Wall-Clock Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

method_colours = {
    'GA':        ('red',        '-'),
    'PSO':       ('blue',       '--'),
    'DE/rand/1': ('green',      '-.'),
    'Memetic DE':('purple',     ':'),
}

# ── Top-left: Convergence curves ────────────────────────────────────────────────
ax = axes[0, 0]
for name, (colour, ls) in method_colours.items():
    r = results[name]
    ax.plot(r['history'], color=colour, linestyle=ls, linewidth=2, label=name)
ax.axhline(BASELINE_ACC, color='gray', linestyle=':', linewidth=1.5, label='Baseline')
ax.set_xlabel('Generation / Iteration', fontsize=11)
ax.set_ylabel('Best 3-fold CV accuracy', fontsize=11)
ax.set_title('Convergence Curves', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Top-right: Final accuracy comparison ────────────────────────────────────────
ax2 = axes[0, 1]
names = list(results.keys())
accs = [results[n]['acc'] for n in names]
stds = [results[n]['std'] for n in names]
n_feats = [results[n]['n_features'] for n in names]
colours_bar = ['gray', 'red', 'blue', 'green', 'purple']

bars = ax2.bar(names, accs, yerr=stds, color=colours_bar, alpha=0.75,
               capsize=5, edgecolor='black')
for bar, nf in zip(bars, n_feats):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{nf}\nfeats', ha='center', va='bottom', fontsize=8)

ax2.set_ylim(0.90, 1.0)
ax2.set_ylabel('5-fold CV accuracy', fontsize=11)
ax2.set_title('Final Accuracy (mean ± std)', fontsize=12)
ax2.tick_params(axis='x', rotation=15)
ax2.grid(True, alpha=0.3, axis='y')

# ── Bottom-left: Wall-clock time ────────────────────────────────────────────────
ax3 = axes[1, 0]
timed_methods = {n: r for n, r in results.items() if r['wall_time'] is not None}
ax3.barh(
    list(timed_methods.keys()),
    [r['wall_time'] for r in timed_methods.values()],
    color=['red', 'blue', 'green', 'purple'],
    alpha=0.75, edgecolor='black'
)
ax3.set_xlabel('Wall-clock time (seconds)', fontsize=11)
ax3.set_title('Runtime Comparison', fontsize=12)
ax3.grid(True, alpha=0.3, axis='x')

# ── Bottom-right: Feature count over time ───────────────────────────────────────
ax4 = axes[1, 1]
for name, (colour, ls) in method_colours.items():
    r = results[name]
    fh = {
        'GA': ga_result['feature_count_history'],
        'PSO': pso_result['feature_count_history'],
        'DE/rand/1': de_result['feature_count_history'],
        'Memetic DE': memetic_result['feature_count_history'],
    }[name]
    ax4.plot(fh, color=colour, linestyle=ls, linewidth=2, label=name)
ax4.axhline(N_FEATURES, color='gray', linestyle=':', linewidth=1.5, label='All features')
ax4.set_xlabel('Generation / Iteration', fontsize=11)
ax4.set_ylabel('Features in best solution', fontsize=11)
ax4.set_title('Feature Count Evolution', fontsize=12)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.suptitle('Benchmark: DE vs Memetic DE vs GA vs PSO\nBreast Cancer Dataset (569 samples, 30 features)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('de_memetic_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: de_memetic_benchmark.png")

## 6. Local Search Contribution Analysis

In [ ]:
# How much did local search contribute to memetic DE?
ls_improvements = memetic_result['ls_improvements']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(range(len(ls_improvements)), ls_improvements,
       color='purple', alpha=0.7, edgecolor='black')
ax.set_xlabel('Generation', fontsize=11)
ax.set_ylabel('Number of bit-flips accepted', fontsize=11)
ax.set_title('Local Search Activity per Generation\n(Memetic DE)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
# Convergence: DE vs Memetic DE
ax2.plot(
    [1 - h for h in de_result['fitness_history']],
    'g-', linewidth=2, label='Pure DE'
)
ax2.plot(
    [1 - h for h in memetic_result['fitness_history']],
    'purple', linewidth=2, linestyle=':', label='Memetic DE'
)
ax2.set_xlabel('Generation', fontsize=11)
ax2.set_ylabel('Best accuracy (3-fold CV)', fontsize=11)
ax2.set_title('DE vs Memetic DE: Convergence Detail', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('memetic_ls_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Local search summary:")
print(f"  Total bit-flips accepted: {sum(ls_improvements)}")
print(f"  Generations with LS improvements: {sum(x > 0 for x in ls_improvements)}/{len(ls_improvements)}")
print(f"  Max LS activity: {max(ls_improvements)} flips in generation {np.argmax(ls_improvements)}")

## 7. DE Strategy Comparison

In [ ]:
strategies = ['rand/1', 'best/1', 'current-to-best/1']
strategy_results = {}

print("Comparing DE mutation strategies (20 pop, 30 gen)...")
for strategy in strategies:
    r = de_rand_1(
        pop_size=20, n_generations=30, F=0.5, CR=0.7,
        strategy=strategy, random_state=42
    )
    strategy_results[strategy] = r
    print(f"  {strategy:<20}: error={r['best_fitness']:.4f}, "
          f"features={r['best_mask'].sum()}, time={r['wall_time']:.1f}s")

fig, ax = plt.subplots(figsize=(10, 5))
colours = ['green', 'darkorange', 'teal']
for (strategy, r), colour in zip(strategy_results.items(), colours):
    ax.plot(r['fitness_history'], linewidth=2, color=colour, label=f'DE/{strategy}')
ax.axhline(BASELINE_ERR, color='gray', linestyle=':', label='Baseline')
ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('Best 3-fold CV error', fontsize=12)
ax.set_title('DE Mutation Strategy Comparison', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('de_strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nNote: DE/rand/1 = most exploration. DE/best/1 = fastest early convergence.")

## 8. Key Takeaways

1. **DE/rand/1** is a robust, parameter-friendly algorithm for feature selection. The continuous relaxation (evolve in $\mathbb{R}^D$, convert to binary via sigmoid) allows DE's differential mutation to operate naturally.

2. **Memetic DE** adds hill-climbing local search to DE. Lamarckian learning (updating the chromosome to the locally improved version) accelerates convergence. The local search contribution is highest in early generations when individuals are far from local optima.

3. **Wall-clock time**: memetic DE takes longer per generation due to local search, but often reaches the same quality in fewer total evaluations. The cache is critical for avoiding redundant evaluations.

4. **Mutation strategy matters**: `DE/rand/1` provides the most exploration (best for multimodal landscapes). `DE/best/1` converges fastest but risks premature convergence. `DE/current-to-best/1` is a practical compromise.

5. **No single algorithm wins**: the benchmark shows all four methods (GA, PSO, DE, Memetic DE) reach comparable accuracy on this dataset. The differences are in feature count, runtime, and convergence speed. Real-world choice depends on available compute budget and problem structure.

---

**References**:
- Storn & Price (1997). Differential Evolution. *Journal of Global Optimization*, 11(4), 341–359.
- Moscato (1989). On Evolution, Search, Optimization and Genetic Algorithms. Caltech Tech Report.
- Brest et al. (2006). Self-adapting control parameters in DE (jDE). *IEEE TEC*, 10(6), 646–657.